In [1]:
import pandas as pd
import numpy as np
from pandas.tseries.offsets import MonthEnd

In [7]:
df = pd.read_csv('sp500_ibes_recommendations_2020_2024.csv')

green_vars = pd.read_csv("../green cleaned.csv", dtype={'ncusip': 'string'})
green_vars['ret_fwd_1'] = (green_vars.groupby('permno')['ret_excess'].shift(-1) )
green_vars = green_vars.dropna(subset=["ret_fwd_1"])

green_vars['datadate'] = pd.to_datetime(green_vars['DATE'], format='%Y%m%d')+pd.tseries.offsets.MonthEnd(0)
green_vars = green_vars[['permno','datadate', 'mve_m', 'ret_fwd_1']]

In [18]:
import pandas as pd
import numpy as np
from pandas.tseries.offsets import MonthEnd

# 1. PREPARE RECOMMENDATION DATA
df['anndats'] = pd.to_datetime(df['anndats'])

# Map IBES codes to zero-centered conviction scores (-2 to 2)
def get_conviction_score(ireccd):
    mapping = {1: 1, 2: 1, 3: 0, 4: -1, 5: -1}
    return mapping.get(ireccd, 0)

df['conviction_score'] = df['ireccd'].apply(get_conviction_score)

# 2. PREPARE MARKET RETURNS (from green_vars)
green_vars['datadate'] = pd.to_datetime(green_vars['datadate'])

market_returns = []
for date in sorted(green_vars['datadate'].unique()):
    date_data = green_vars[green_vars['datadate'] == date].dropna(subset=['ret_fwd_1', 'mve_m'])
    if len(date_data) > 0:
        weights = date_data['mve_m'] / date_data['mve_m'].sum()
        mkt_ret = (weights * date_data['ret_fwd_1']).sum()
        market_returns.append({'datadate': date, 'mkt_ret': mkt_ret})

mkt_df = pd.DataFrame(market_returns).set_index('datadate')

# 3. ROLLING WINDOW SETUP
train_window_months = 12
oos_start = pd.to_datetime('2021-10-31') + MonthEnd(0)
all_oos_dates = sorted([d for d in green_vars['datadate'].unique() if d >= oos_start])

all_oos_recommendations = []

print("Starting Out-of-Sample Rolling Windows...")

for oos_date in all_oos_dates:
    print(oos_date)
    train_end = oos_date - pd.DateOffset(months=1) + MonthEnd(0)
    train_start = train_end - pd.DateOffset(months=train_window_months) + MonthEnd(0)
    
    # --- A. TRAINING PHASE (Calculate Alphas) ---
    train_recs = df[(df['anndats'] >= train_start) & (df['anndats'] <= train_end)].copy()
    analyst_alphas = {}
    
    for analyst_id, analyst_group in train_recs.groupby('emaskcd'):
        analyst_returns = []
        train_months = pd.date_range(start=train_start, end=train_end, freq='ME')
        
        for t_month in train_months:
            active_recs = analyst_group[(analyst_group['anndats'] <= t_month) & 
                                        (analyst_group['anndats'] > t_month - pd.Timedelta(days=30))]
            if len(active_recs) == 0: 
                continue
                
            month_returns = green_vars[green_vars['datadate'] == t_month]
            merged = pd.merge(active_recs, month_returns, on='permno', how='inner')
            
            if len(merged) > 0:
                port_ret = (merged['conviction_score'] * merged['ret_fwd_1']).mean()
                analyst_returns.append(port_ret)
        
        if len(analyst_returns) > 0:
            avg_analyst_ret = np.mean(analyst_returns)
            mkt_slice = mkt_df[(mkt_df.index >= train_start) & (mkt_df.index <= train_end)]['mkt_ret'].mean()
            alpha = avg_analyst_ret - mkt_slice
            
            if alpha > 0: # Only keep positive alpha analysts
                analyst_alphas[analyst_id] = alpha

    # --- B. RECOMMENDATION GENERATION PHASE ---
    if not analyst_alphas:
        continue
        
    lookback_start = oos_date - pd.Timedelta(days=30)
    current_recs = df[(df['anndats'] > lookback_start) & (df['anndats'] <= oos_date) & 
                      (df['emaskcd'].isin(analyst_alphas.keys()))].copy()
    
    if len(current_recs) == 0:
        continue

    # Map raw alphas directly to the dataframe
    current_recs['alpha_weight'] = current_recs['emaskcd'].map(analyst_alphas)
    
    # Custom function to calculate weighted average directly per stock
    def get_stock_weighted_average(group):
        total_weight = group['alpha_weight'].sum()
        if total_weight == 0:
            return 0
        return (group['conviction_score'] * group['alpha_weight']).sum() / total_weight

    stock_signals = current_recs.groupby('permno').apply(get_stock_weighted_average, include_groups=False)
    ticker_map = current_recs.drop_duplicates('permno').set_index('permno')['ticker_crsp'].to_dict()
    
    # Discretize into BUY / HOLD / SELL based on normalized [-2, 2] scale
    for permno, w_score in stock_signals.items():
        if w_score >= 0.5:
            action = 'BUY'
        elif w_score <= -0.5:
            action = 'SELL'
        else:
            action = 'HOLD'
            
        all_oos_recommendations.append({
            'permno': permno,
            'date': oos_date.strftime('%Y-%m-%d'),
            'ticker': ticker_map.get(permno, 'UNKNOWN'),
            'weighted_mean_recommendation': w_score,
            'action': action
        })

# --- C. FINALIZE DATAFRAME ---
final_recs_df = pd.DataFrame(all_oos_recommendations)
final_recs_df = final_recs_df.sort_values(by=['permno', 'date']).reset_index(drop=True)

print("\nFinal Out-of-Sample Recommendations:")
print(final_recs_df.head(20))

final_recs_df.to_csv('smart_analyst_recommendations.csv', index=False)

Starting Out-of-Sample Rolling Windows...
2021-10-31 00:00:00
2021-11-30 00:00:00
2021-12-31 00:00:00
2022-01-31 00:00:00
2022-02-28 00:00:00
2022-03-31 00:00:00
2022-04-30 00:00:00
2022-05-31 00:00:00
2022-06-30 00:00:00
2022-07-31 00:00:00
2022-08-31 00:00:00
2022-09-30 00:00:00
2022-10-31 00:00:00
2022-11-30 00:00:00
2022-12-31 00:00:00
2023-01-31 00:00:00
2023-02-28 00:00:00
2023-03-31 00:00:00
2023-04-30 00:00:00
2023-05-31 00:00:00
2023-06-30 00:00:00
2023-07-31 00:00:00
2023-08-31 00:00:00
2023-09-30 00:00:00
2023-10-31 00:00:00
2023-11-30 00:00:00
2023-12-31 00:00:00
2024-01-31 00:00:00
2024-02-29 00:00:00
2024-03-31 00:00:00
2024-04-30 00:00:00
2024-05-31 00:00:00
2024-06-30 00:00:00
2024-07-31 00:00:00
2024-08-31 00:00:00
2024-09-30 00:00:00
2024-10-31 00:00:00
2024-11-30 00:00:00

Final Out-of-Sample Recommendations:
    permno        date ticker  weighted_mean_recommendation action
0    10104  2021-12-31   ORCL                      0.000000   HOLD
1    10104  2022-03-31   O